# F7-kernels-convex-optimization — Practice p09

**Type:** constrained coding · **Difficulty:** core · **Concepts:** convex-functions

Implement `jensen_gaps(f, X, Y, theta)` for a vectorized real function `f`.

Input contract: `X` and `Y` are finite real-numeric NumPy arrays with the same nonempty shape; `theta` is a finite real numeric scalar in `[0,1]`; and `f` is callable. Accept Python or NumPy integer/floating scalars, including the endpoints; reject booleans, text, bytes, and containers even if they could be converted to a float. The three calls `f(X)`, `f(Y)`, and `f((1-theta)*X + theta*Y)` must each produce a finite real-numeric NumPy array with exactly the input shape. Reject any violation with `ValueError`; do not mutate `X` or `Y`, even if `f` mutates the array it receives. Pass a separate defensive copy to each call of `f`.

Return the finite floating NumPy array

$$(1-\theta)f(X)+\theta f(Y)-f((1-\theta)X+\theta Y)$$

with the same shape. Use `ATOL = 1e-10`, `RTOL = 0.0`. Nonnegative entries are Jensen evidence for the tested pairs; one entry below `-ATOL` is a finite convexity counterexample.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

def jensen_gaps(f, X, Y, theta):
    """Compute coordinatewise Jensen gaps using defensive function inputs."""
    if (
        not callable(f)
        or not isinstance(X, np.ndarray)
        or not isinstance(Y, np.ndarray)
        or X.shape != Y.shape
        or X.size == 0
        or not np.issubdtype(X.dtype, np.number)
        or not np.issubdtype(Y.dtype, np.number)
        or np.iscomplexobj(X)
        or np.iscomplexobj(Y)
        or not np.isfinite(X).all()
        or not np.isfinite(Y).all()
    ):
        raise ValueError("f, X, and Y do not satisfy the input contract")
    if (
        isinstance(theta, (bool, np.bool_))
        or not isinstance(theta, (int, float, np.integer, np.floating))
    ):
        raise ValueError("theta must be a Python or NumPy integer/floating scalar")
    try:
        theta_float = float(theta)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("theta must be a finite real scalar") from exc
    if not np.isfinite(theta_float) or not 0.0 <= theta_float <= 1.0:
        raise ValueError("theta must lie in [0, 1]")

    X_private = X.copy()
    Y_private = Y.copy()
    mixed = (1.0 - theta_float) * X_private + theta_float * Y_private

    try:
        raw_fx = f(X_private.copy())
        raw_fy = f(Y_private.copy())
        raw_fmixed = f(mixed.copy())
    except Exception as exc:
        raise ValueError("f must accept each defensive array copy") from exc

    values = []
    for raw in (raw_fx, raw_fy, raw_fmixed):
        if (
            not isinstance(raw, np.ndarray)
            or raw.shape != X.shape
            or not np.issubdtype(raw.dtype, np.number)
            or np.iscomplexobj(raw)
            or not np.isfinite(raw).all()
        ):
            raise ValueError("every function result must match the finite real input shape")
        values.append(raw.astype(float, copy=True))
    fx, fy, fmixed = values
    gaps = (1.0 - theta_float) * fx + theta_float * fy - fmixed
    if not np.isfinite(gaps).all():
        raise ValueError("the floating Jensen computation must remain finite")
    return gaps

## Immutable contract check — do not edit

The public fixtures include convex and nonconvex functions, different shapes and weights, exact formula checks, rejection, and non-mutation. A checker does not infer universal convexity from sampled gaps.

In [ ]:
def _square_shift_p09(t):
    return (t + 1.0) ** 2

def _absolute_p09(t):
    return np.abs(t)

def _concave_p09(t):
    return -(t ** 2)

_cases_p09 = (
    (_square_shift_p09, np.array([-2.0, 1.0]), np.array([4.0, -3.0]), np.int64(0)),
    (_square_shift_p09, np.array([-2.0, 1.0]), np.array([4.0, -3.0]), 1),
    (_square_shift_p09, np.array([-3.0, 0.0, 2.0]), np.array([5.0, 4.0, -1.0]), 0.25),
    (_absolute_p09, np.array([[-2.0, 1.0], [3.0, -4.0]]), np.array([[4.0, -3.0], [-1.0, 2.0]]), 0.5),
    (_concave_p09, np.array([-2.0, 0.0, 3.0]), np.array([4.0, 1.0, -1.0]), 0.6),
)
for _f_p09, _X_p09, _Y_p09, _theta_p09 in _cases_p09:
    _before_X_p09 = _X_p09.copy()
    _before_Y_p09 = _Y_p09.copy()
    _gaps_p09 = jensen_gaps(_f_p09, _X_p09, _Y_p09, _theta_p09)
    _mix_p09 = (1.0 - _theta_p09) * _X_p09 + _theta_p09 * _Y_p09
    _expected_p09 = (
        (1.0 - _theta_p09) * _f_p09(_X_p09)
        + _theta_p09 * _f_p09(_Y_p09)
        - _f_p09(_mix_p09)
    )
    assert np.array_equal(_X_p09, _before_X_p09)
    assert np.array_equal(_Y_p09, _before_Y_p09)
    assert isinstance(_gaps_p09, np.ndarray) and _gaps_p09.shape == _X_p09.shape
    assert np.issubdtype(_gaps_p09.dtype, np.floating) and np.isfinite(_gaps_p09).all()
    assert np.allclose(_gaps_p09, _expected_p09, atol=ATOL, rtol=RTOL)

def _mutating_square_p09(t):
    original = t.copy()
    t[...] = 12345.0
    return original ** 2

_X_mut_p09 = np.array([-3.0, 1.0, 2.0])
_Y_mut_p09 = np.array([4.0, -2.0, 5.0])
_X_mut_before_p09 = _X_mut_p09.copy()
_Y_mut_before_p09 = _Y_mut_p09.copy()
_theta_mut_p09 = 0.3
_mut_gaps_p09 = jensen_gaps(_mutating_square_p09, _X_mut_p09, _Y_mut_p09, _theta_mut_p09)
_mix_mut_p09 = (1.0 - _theta_mut_p09) * _X_mut_before_p09 + _theta_mut_p09 * _Y_mut_before_p09
_expected_mut_p09 = (
    (1.0 - _theta_mut_p09) * _X_mut_before_p09 ** 2
    + _theta_mut_p09 * _Y_mut_before_p09 ** 2
    - _mix_mut_p09 ** 2
)
assert np.array_equal(_X_mut_p09, _X_mut_before_p09)
assert np.array_equal(_Y_mut_p09, _Y_mut_before_p09)
assert np.allclose(_mut_gaps_p09, _expected_mut_p09, atol=ATOL, rtol=RTOL)

_X0_p09 = np.array([0.0, 1.0])
_Y0_p09 = np.array([2.0, 3.0])
_invalid_p09 = (
    (lambda t: t, [0.0], np.array([1.0]), 0.5),
    (lambda t: t, _X0_p09, np.array([[2.0, 3.0]]), 0.5),
    (lambda t: t, np.array([]), np.array([]), 0.5),
    (lambda t: t, _X0_p09, _Y0_p09, -0.1),
    (lambda t: t, _X0_p09, _Y0_p09, 1.1),
    (lambda t: t, _X0_p09, _Y0_p09, np.nan),
    (lambda t: t, _X0_p09, _Y0_p09, "0.5"),
    (lambda t: t, _X0_p09, _Y0_p09, b"0.5"),
    (lambda t: t, _X0_p09, _Y0_p09, True),
    (lambda t: t, _X0_p09, _Y0_p09, np.bool_(False)),
    (7, _X0_p09, _Y0_p09, 0.5),
    (lambda t: 1.0, _X0_p09, _Y0_p09, 0.5),
    (lambda t: np.full(t.shape, np.nan), _X0_p09, _Y0_p09, 0.5),
    (lambda t: t, _X0_p09.astype(complex), _Y0_p09, 0.5),
)
for _args_p09 in _invalid_p09:
    try:
        jensen_gaps(*_args_p09)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid Jensen data must raise ValueError")

### Solution reasoning

The mixture is formed from private copies before calling the function. Each of the three evaluations receives a different defensive copy, so even a mutating callable cannot alter the caller's \(X\) or \(Y\), the mixture used elsewhere, or another evaluation. Output validation is performed before combining values. The final expression is exactly the right side of Jensen's inequality minus the left side, so nonnegative coordinates support convexity for those tested pairs while a negative coordinate supplies a counterexample.

### Answer check

In [ ]:
_X_answer_p09 = np.array([-2.0, 1.0])
_Y_answer_p09 = np.array([4.0, -3.0])
_theta_answer_p09 = 0.25
_gaps_answer_p09 = jensen_gaps(
    lambda t: t ** 2,
    _X_answer_p09,
    _Y_answer_p09,
    _theta_answer_p09,
)
_expected_answer_p09 = (
    _theta_answer_p09
    * (1.0 - _theta_answer_p09)
    * (_X_answer_p09 - _Y_answer_p09) ** 2
)
assert np.allclose(
    _gaps_answer_p09,
    _expected_answer_p09,
    atol=ATOL,
    rtol=RTOL,
)